# 09 · Category as a live feature (task 5.14)

Can a small classifier on the pipeline's title vectors give each trend the category the local model drafts and the check confirms? If so, the pipeline can write a topic's category every hour without a language model.

**Data.** The labeled trends in `research/data/categories.csv` that the pipeline's filters keep, with their vectors from the pipeline's own model (`scripts/embed-titles.ts`).

**Method** (`topnews.category_model`).
- The newest 30% of trends by first sighting are the test set.
- The classifier is a logistic regression on the 384-number vector, with or without the platform. Its options and the confidence it needs before answering are chosen inside the training period, by forward-chaining folds.
- It trains on every label and is scored only on checked labels, against the checked label.
- It may answer "unknown". The targets (TASKS.md 5.14): right on 85% of what it answers, and an answer for at least half of the trends.

**Needs** `npm ci` at the repo root (as notebook 06 does). Run top to bottom. Clear the outputs before committing (research/README.md).

In [ ]:
import pandas as pd

from topnews import category_model as cm
from topnews import db

engine = db.engine()  # read-only; RESEARCH_DATABASE_URL from .env.local
items = db.trend_items(engine, days=28)  # YouTube left out
table, X = cm.training_set(items)
test = cm.time_split(table) & table.checked.to_numpy()
train = ~cm.time_split(table)
src, y = table.source_id.to_numpy(), table.category.to_numpy()
print(f"{len(table)} labeled trends, {int(table.checked.sum())} checked; train {int(train.sum())} "
      f"(to {table.first_seen[train].max():%m-%d %H:%M} UTC), test {int(test.sum())}")

## What it has to beat

Each platform's most common category, the nearest labeled trend, and (as the ceiling) the local model's own draft.

In [ ]:
simple = cm.baselines(X[train], src[train], y[train], X[test], src[test], y[test])
simple["local_model_draft"] = float((table.llm_category[test] == table.category[test]).mean())
pd.Series(simple).round(3)

## The classifier

Chosen in training, then scored on the test set: always answering, and at its threshold.

In [ ]:
model = cm.choose(X[train], src[train], y[train])
print(model.meta, "threshold", model.threshold)
always, _ = cm.predict(cm.with_threshold(model, 0.0), X[test], src[test])
gated, _ = cm.predict(model, X[test], src[test])
pd.DataFrame({"always answering": cm.score(always, y[test]), "at its threshold": cm.score(gated, y[test])}).round(3)

## Coverage against accuracy

A stricter threshold answers fewer trends and gets more of them right.

In [ ]:
rows = []
for threshold in (0.0, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9):
    answers, _ = cm.predict(cm.with_threshold(model, threshold), X[test], src[test])
    rows.append({"threshold": threshold, **cm.score(answers, y[test])})
pd.DataFrame(rows).round(3)

## By platform and by category

Always answering, beside the answers at the threshold.

In [ ]:
display(cm.by_group(always, y[test], src[test]).join(cm.by_group(gated, y[test], src[test]), rsuffix="_at_threshold").round(2))
cm.by_group(always, y[test], y[test]).join(cm.by_group(gated, y[test], y[test]), rsuffix="_at_threshold").round(2)

## Do more labels help?

Training on only the newest n training trends, scored on the same test set.

In [ ]:
sizes = [n for n in (65, 130, 260, 390, 520, 650, 1000, 1500, 2000, 3000) if n < train.sum()] + [int(train.sum())]
options = {"c": model.meta["c"], "use_platform": model.meta["use_platform"]}
cm.learning_curve(X[train], src[train], y[train], X[test], src[test], y[test], sizes, **options).round(3)

## The file the pipeline would load

Not shipped until the targets are met. The JSON holds the classes, the platforms, the weights, the bias and the threshold; the pipeline's score is one matrix product and a softmax.

In [ ]:
print(len(cm.to_json(model)) // 1024, "KB")